# 05 — Evaluating a Target Application

**What you'll learn:**
- Using `target=` to score a *live* function instead of pre-computed responses
- How `evaluate()` calls your app row-by-row
- `column_mapping` to wire dataset/target fields into evaluators

---

So far our data already contained a `response` column. In reality you want to test
**your app** as it changes. Pass a `target` callable and `evaluate()` will run it on
each row's inputs, capture the output, then score it. This is what you gate a PR on.

In [ ]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

## The target function

A target takes the dataset columns as keyword args and returns a dict. Here we stub a
tiny "insurance assistant" — swap this for a real Azure OpenAI / Agent call in your app.

In [ ]:
def insurance_assistant(query: str, **kwargs) -> dict:
    """Stand-in for your real app. Replace with an actual model/agent call."""
    canned = {
        "deadline": "Most policies require filing an auto claim within 30 days.",
        "deductible": "A deductible is the amount you pay before coverage applies.",
    }
    text = next((v for k, v in canned.items() if k in query.lower()), 
                "Please refer to your policy documents for details.")
    return {"response": text}


insurance_assistant(query="What is a deductible?")

## Run evaluate() with a target

We give `evaluate()` a dataset of **queries only** (plus `ground_truth`). The `target`
produces `response` for each row; evaluators then score that generated response.

In [ ]:
from azure.ai.evaluation import evaluate, F1ScoreEvaluator, RelevanceEvaluator

result = evaluate(
    data=f"{DATA_DIR}/insurance-qa.jsonl",
    target=insurance_assistant,
    evaluators={
        "f1": F1ScoreEvaluator(),
        "relevance": RelevanceEvaluator(model_config),
    },
)
result["metrics"]

## Column mapping

When your dataset or target uses different field names than an evaluator expects,
wire them explicitly with `evaluator_config`. References:

- `${data.<col>}` — a column from the dataset file
- `${target.<key>}` — a key returned by the target function

Below we make it explicit that `response` comes from the target and `ground_truth`
comes from the data.

In [ ]:
result = evaluate(
    data=f"{DATA_DIR}/insurance-qa.jsonl",
    target=insurance_assistant,
    evaluators={"f1": F1ScoreEvaluator()},
    evaluator_config={
        "f1": {
            "column_mapping": {
                "response": "${target.response}",
                "ground_truth": "${data.ground_truth}",
            }
        }
    },
)
result["metrics"]

## Recap

- `target=` lets you evaluate a live app; `evaluate()` runs it per row.
- Evaluators then score the *generated* output — the basis of a CI quality gate.
- `column_mapping` with `${data.*}` / `${target.*}` wires fields to evaluator inputs.

**Next:** [06 — Cloud & Foundry evaluation](./06-cloud-foundry-evaluation.ipynb) — log
and view results in the Foundry portal.